In [ ]:
# ===== Experiment selection =====
# Run one profile per Colab session.
# Options: "mistral24b", "qwen30b", "phi4"
MODEL_PROFILE = "mistral24b"

# Keep these as close as possible to the Gemma 4 26B k=20 validation setup.
SPLIT = "validation"
LANG = None
LIMIT = None
OFFSET = 0
RESUME = True

# Local + Drive output. Drive survives Colab runtime resets.
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/MultiLexNorm2026/llm_detection_experiments"

# If T4 OOM happens, try CONTEXT_OVERRIDE = 8192 first.
CONTEXT_OVERRIDE = None
GPU_LAYERS_OVERRIDE = None

REPO_URL = "https://github.com/jaeiko/MultiLexNorm_HW11.git"
BRANCH = "detection-model"


In [ ]:
# ===== Drive / GitHub / Hugging Face token setup =====
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    github_token = userdata.get("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN")
    hf_token = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN")
    hf_token = os.environ.get("HF_TOKEN")

repo_dir = Path("/content/MultiLexNorm_HW11")
if repo_dir.exists():
    subprocess.check_call(["git", "-C", str(repo_dir), "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(repo_dir), "checkout", BRANCH])
    subprocess.check_call(["git", "-C", str(repo_dir), "pull", "origin", BRANCH])
else:
    clone_url = REPO_URL
    if github_token:
        clone_url = REPO_URL.replace("https://", f"https://{quote(github_token, safe='')}@")
    subprocess.check_call(["git", "clone", "-b", BRANCH, clone_url, str(repo_dir)])

candidates = list(repo_dir.glob("*/LLM-base*"))
if not candidates:
    raise FileNotFoundError("LLM-base detection folder was not found.")
detector_dir = candidates[0]
sys.path.insert(0, str(detector_dir))

drive_output_root = Path(DRIVE_OUTPUT_ROOT)
drive_output_root.mkdir(parents=True, exist_ok=True)

print("repo_dir:", repo_dir)
print("detector_dir:", detector_dir)
print("drive_output_root:", drive_output_root)


In [ ]:
# ===== Dependency installation =====
import subprocess
import sys

def pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip_install(["-U", "pandas", "pyarrow", "huggingface_hub", "tqdm"])

from colab_multi_model_runner import get_model_profile
profile = get_model_profile(MODEL_PROFILE)

if profile["provider"] == "llama_cpp":
    import torch
    cuda_version = torch.version.cuda or "12.1"
    cuda_tag = "cu124" if cuda_version.startswith(("12.4", "12.5", "12.6")) else "cu121"
    wheel_index = f"https://abetlen.github.io/llama-cpp-python/whl/{cuda_tag}"
    try:
        pip_install(["-U", "llama-cpp-python", "--extra-index-url", wheel_index])
    except subprocess.CalledProcessError:
        print("CUDA wheel install failed. Building llama-cpp-python with GGML_CUDA=on...")
        os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
        os.environ["FORCE_CMAKE"] = "1"
        pip_install(["--no-cache-dir", "--force-reinstall", "-U", "llama-cpp-python"])

print("Dependencies are ready for", MODEL_PROFILE)


In [ ]:
# ===== Run experiment =====
from colab_multi_model_runner import (
    build_client,
    create_experiment_configs,
    get_model_profile,
    run_experiment_streaming,
)

profile = get_model_profile(MODEL_PROFILE)
if CONTEXT_OVERRIDE is not None:
    profile["num_ctx"] = int(CONTEXT_OVERRIDE)
if GPU_LAYERS_OVERRIDE is not None:
    profile["n_gpu_layers"] = int(GPU_LAYERS_OVERRIDE)

experiment_dir = create_experiment_configs(
    detector_dir,
    profile,
    split=SPLIT,
    lang=LANG,
    limit=LIMIT,
    offset=OFFSET,
)
drive_experiment_dir = Path(DRIVE_OUTPUT_ROOT) / profile["experiment_name"]

print("Local experiment dir:", experiment_dir)
print("Drive experiment dir:", drive_experiment_dir)

client = build_client(profile, cache_dir="/content/model_cache")
metrics = run_experiment_streaming(
    detector_dir,
    experiment_dir,
    client,
    resume=RESUME,
    mirror_dir=drive_experiment_dir,
)


In [ ]:
# ===== Inspect result =====
from pathlib import Path

benchmark_path = Path(experiment_dir) / "benchmark.txt"
summary_path = Path(experiment_dir) / "summary.md"
drive_benchmark_path = Path(DRIVE_OUTPUT_ROOT) / profile["experiment_name"] / "benchmark.txt"

print("local experiment_dir:", experiment_dir)
print("drive experiment_dir:", Path(DRIVE_OUTPUT_ROOT) / profile["experiment_name"])
if benchmark_path.exists():
    print("\n--- local benchmark.txt ---")
    print(benchmark_path.read_text(encoding="utf-8"))
if drive_benchmark_path.exists():
    print("\n--- drive benchmark.txt ---")
    print(drive_benchmark_path.read_text(encoding="utf-8"))
if summary_path.exists():
    print("\n--- summary.md ---")
    print(summary_path.read_text(encoding="utf-8"))
